In [ ]:
# Cell 1: Import necessary libraries
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set random seed
random_state = 6740
np.random.seed(random_state)

In [ ]:
# Cell 2: Function to load and preprocess MNIST Digits (from .mat)
# Cell 2: Function to load and preprocess MNIST Digits (from .mat)
def load_and_preprocess_mnist_digits(filepath):
    mat = scipy.io.loadmat(filepath)
    # The keys 'X_train', 'y_train', 'X_test', 'y_test' were placeholders.
    # Now confirmed to be 'xtrain', 'ytrain', 'xtest', 'ytest'
    xtrain = mat['xtrain'].astype(np.float32) # Using 'xtrain' as per your provided structure
    ytrain = mat['ytrain'].flatten()         # Using 'ytrain' as per your provided structure
    xtest = mat['xtest'].astype(np.float32)  # Using 'xtest' as per your provided structure
    ytest = mat['ytest'].flatten()           # Using 'ytest' as per your provided structure

    # Standardize data to [0, 1]
    xtrain = xtrain / 255.0
    xtest = xtest / 255.0

    return xtrain, ytrain, xtest, ytest

# Load MNIST Digits
print("Loading MNIST Digits...")
# Ensure the path is correct relative to your script
xtrain_digits, ytrain_digits, xtest_digits, ytest_digits = load_and_preprocess_mnist_digits('data/mnist_10digits.mat')
print(f"MNIST Digits - xtrain shape: {xtrain_digits.shape}, ytrain shape: {ytrain_digits.shape}")
print(f"MNIST Digits - xtest shape: {xtest_digits.shape}, ytest shape: {ytest_digits.shape}")

In [ ]:
# Cell 3: Function to load and preprocess Fashion MNIST (from .csv)
def load_and_preprocess_fashion_mnist(train_filepath, test_filepath):
    # Load training data
    train_df = pd.read_csv(train_filepath)
    xtrain = train_df.drop('label', axis=1).values.astype(np.float32)
    ytrain = train_df['label'].values.flatten()

    # Load test data
    test_df = pd.read_csv(test_filepath)
    xtest = test_df.drop('label', axis=1).values.astype(np.float32)
    ytest = test_df['label'].values.flatten()

    # Standardize data to [0, 1]
    xtrain = xtrain / 255.0
    xtest = xtest / 255.0

    return xtrain, ytrain, xtest, ytest

# Load MNIST Fashion
print("\nLoading MNIST Fashion...")
# Ensure the paths are correct relative to your script
xtrain_fashion, ytrain_fashion, xtest_fashion, ytest_fashion = load_and_preprocess_fashion_mnist(
    'data/fashion-mnist_train.csv', 'data/fashion-mnist_test.csv'
)
print(f"MNIST Fashion - xtrain shape: {xtrain_fashion.shape}, ytrain shape: {ytrain_fashion.shape}")
print(f"MNIST Fashion - xtest shape: {xtest_fashion.shape}, ytest shape: {ytest_fashion.shape}")

In [ ]:
# Cell 4: Train and evaluate Logistic Regression
def train_and_evaluate_logistic_regression(xtrain, ytrain, xtest, ytest, dataset_name):
    print(f"\n--- Training Logistic Regression on {dataset_name} ---")
    # Increased max_iter for better convergence on larger datasets
    log_reg = LogisticRegression(solver='saga', multi_class='multinomial', max_iter=50, random_state=random_state, n_jobs=-1)
    log_reg.fit(xtrain, ytrain)
    y_pred = log_reg.predict(xtest)
    report = classification_report(ytest, y_pred, output_dict=True, zero_division=0)
    print(f"Classification Report for Logistic Regression on {dataset_name}:\n")
    print(pd.DataFrame(report).transpose())
    return report

log_reg_digits_report = train_and_evaluate_logistic_regression(xtrain_digits, ytrain_digits, xtest_digits, ytest_digits, "MNIST Digits")
log_reg_fashion_report = train_and_evaluate_logistic_regression(xtrain_fashion, ytrain_fashion, xtest_fashion, ytest_fashion, "MNIST Fashion")

In [ ]:
# Cell 5: Train and evaluate K-Nearest Neighbors
def train_and_evaluate_knn(xtrain, ytrain, xtest, ytest, dataset_name):
    print(f"\n--- Training K-Nearest Neighbors on {dataset_name} ---")
    # For KNN, K=5 is a reasonable starting point.
    # To obtain the "best K", one would typically use cross-validation on a smaller subset
    # or a validation set, and try a range of K values.
    # Given the scope, and computational cost for 60k samples, we'll fix K to a common good value.
    best_k = 5
    knn = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
    knn.fit(xtrain, ytrain)
    y_pred = knn.predict(xtest)
    report = classification_report(ytest, y_pred, output_dict=True, zero_division=0)
    print(f"Classification Report for KNN (K={best_k}) on {dataset_name}:\n")
    print(pd.DataFrame(report).transpose())
    return report, best_k

knn_digits_report, k_digits = train_and_evaluate_knn(xtrain_digits, ytrain_digits, xtest_digits, ytest_digits, "MNIST Digits")
knn_fashion_report, k_fashion = train_and_evaluate_knn(xtrain_fashion, ytrain_fashion, xtest_fashion, ytest_fashion, "MNIST Fashion")

In [ ]:
# Cell 6: Train and evaluate SVM (Linear and Kernel RBF)
def train_and_evaluate_svm(xtrain, ytrain, xtest, ytest, dataset_name):
    print(f"\n--- Training SVM on {dataset_name} (Downsampled Training Data for SVC) ---")
    # Randomly downsample training data to m=5000 for efficiency.
    m = 5000
    if xtrain.shape[0] > m:
        sample_indices = np.random.choice(xtrain.shape[0], m, replace=False)
        xtrain_sampled = xtrain[sample_indices]
        ytrain_sampled = ytrain[sample_indices]
    else:
        xtrain_sampled = xtrain # Use full data if smaller than m
        ytrain_sampled = ytrain

    print(f"Using {xtrain_sampled.shape[0]} samples for SVM training.")
    # Linear SVM
    print(f"\n--- Training Linear SVM on {dataset_name} ---")
    # dual=False is recommended for n_samples > n_features for 'linear' kernel and 'l2' penalty
    svm_linear = SVC(kernel='linear', random_state=random_state, verbose=False)
    svm_linear.fit(xtrain_sampled, ytrain_sampled)
    y_pred_linear = svm_linear.predict(xtest)
    report_linear = classification_report(ytest, y_pred_linear, output_dict=True, zero_division=0)
    print(f"Classification Report for Linear SVM on {dataset_name}:\n")
    print(pd.DataFrame(report_linear).transpose())

    # Kernel SVM (RBF)
    print(f"\n--- Training Kernel SVM (RBF) on {dataset_name} ---")
    svm_rbf = SVC(kernel='rbf', random_state=random_state, verbose=False)
    svm_rbf.fit(xtrain_sampled, ytrain_sampled)
    y_pred_rbf = svm_rbf.predict(xtest)
    report_rbf = classification_report(ytest, y_pred_rbf, output_dict=True, zero_division=0)
    print(f"Classification Report for Kernel SVM (RBF) on {dataset_name}:\n")
    print(pd.DataFrame(report_rbf).transpose())

    return report_linear, report_rbf

svm_linear_digits_report, svm_rbf_digits_report = train_and_evaluate_svm(xtrain_digits, ytrain_digits, xtest_digits, ytest_digits, "MNIST Digits")
svm_linear_fashion_report, svm_rbf_fashion_report = train_and_evaluate_svm(xtrain_fashion, ytrain_fashion, xtest_fashion, ytest_fashion, "MNIST Fashion")


In [ ]:
# Cell 7: Train and evaluate Neural Networks (MLPClassifier)
def train_and_evaluate_mlp(xtrain, ytrain, xtest, ytest, dataset_name):
    print(f"\n--- Training Neural Network (MLPClassifier) on {dataset_name} ---")
    # Using specified hidden_layer_sizes = (20, 10)
    mlp = MLPClassifier(hidden_layer_sizes=(20, 10), max_iter=200, random_state=random_state, verbose=True)
    mlp.fit(xtrain, ytrain)
    y_pred = mlp.predict(xtest)
    report = classification_report(ytest, y_pred, output_dict=True, zero_division=0)
    print(f"Classification Report for MLPClassifier on {dataset_name}:\n")
    print(pd.DataFrame(report).transpose())
    return report

mlp_digits_report = train_and_evaluate_mlp(xtrain_digits, ytrain_digits, xtest_digits, ytest_digits, "MNIST Digits")
mlp_fashion_report = train_and_evaluate_mlp(xtrain_fashion, ytrain_fashion, xtest_fashion, ytest_fashion, "MNIST Fashion")

In [ ]:
# Cell 8: Consolidate and visualize results (Optional, but good for discussion)

def plot_overall_metrics(reports, metric_name, title):
    data = []
    for model, datasets in reports.items():
        for dataset_name, report in datasets.items():
            value = None
            if metric_name == 'accuracy':
                if 'accuracy' in report:
                    value = report['accuracy']
            else: # For precision, recall, f1-score, look into 'macro avg' or 'weighted avg'
                if 'macro avg' in report and metric_name in report['macro avg']:
                    value = report['macro avg'][metric_name]
                elif 'weighted avg' in report and metric_name in report['weighted avg']:
                    value = report['weighted avg'][metric_name]

            if value is not None:
                data.append({'Model': model, 'Dataset': dataset_name, 'Value': value})
            else:
                print(f"Warning: Could not find '{metric_name}' for {model} on {dataset_name}. Skipping.")


    df = pd.DataFrame(data)
    if df.empty:
        print(f"No data to plot for {title}.")
        return

    plt.figure(figsize=(12, 6))
    sns.barplot(x='Model', y='Value', hue='Dataset', data=df)
    plt.title(title)
    plt.ylabel(metric_name)
    plt.ylim(0, 1) # Metrics like precision, recall, f1-score, accuracy are between 0 and 1
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Collect reports for plotting overall metrics (e.g., accuracy or macro avg F1)
all_reports = {
    "Logistic Regression": {"MNIST Digits": log_reg_digits_report, "MNIST Fashion": log_reg_fashion_report},
    f"KNN (K={k_digits}/{k_fashion})": {"MNIST Digits": knn_digits_report, "MNIST Fashion": knn_fashion_report},
    "Linear SVM": {"MNIST Digits": svm_linear_digits_report, "MNIST Fashion": svm_linear_fashion_report},
    "Kernel SVM (RBF)": {"MNIST Digits": svm_rbf_digits_report, "MNIST Fashion": svm_rbf_fashion_report},
    "Neural Network (MLP)": {"MNIST Digits": mlp_digits_report, "MNIST Fashion": mlp_fashion_report},
}

# Plot Macro Avg F1-score
# No need for the complex comprehension here, pass all_reports directly
plot_overall_metrics(all_reports, 'f1-score', "Macro Avg F1-score for Classifiers on MNIST Datasets")

# Plot Overall Accuracy
# No need for the complex comprehension here, pass all_reports directly
plot_overall_metrics(all_reports, 'accuracy', "Overall Accuracy for Classifiers on MNIST Datasets")

In [ ]:
# Cell 9: Detailed reporting (already done in previous cells, but summarizing structure for clarity)
# The classification_report function already provides precision, recall, and F1-score for each class.
# The previous cells printed these reports.
# To make it explicit for question 1, here's a recap of where to find the answers:

print("--- Detailed Classification Reports ---")

print("\n--- MNIST Digits Reports ---")
print("Logistic Regression (Digits):")
print(pd.DataFrame(log_reg_digits_report).transpose())
print("\nKNN (Digits):")
print(pd.DataFrame(knn_digits_report).transpose())
print("\nLinear SVM (Digits):")
print(pd.DataFrame(svm_linear_digits_report).transpose())
print("\nKernel SVM (RBF) (Digits):")
print(pd.DataFrame(svm_rbf_digits_report).transpose())
print("\nNeural Network (MLP) (Digits):")
print(pd.DataFrame(mlp_digits_report).transpose())


print("\n--- MNIST Fashion Reports ---")
print("Logistic Regression (Fashion):")
print(pd.DataFrame(log_reg_fashion_report).transpose())
print("\nKNN (Fashion):")
print(pd.DataFrame(knn_fashion_report).transpose())
print("\nLinear SVM (Fashion):")
print(pd.DataFrame(svm_linear_fashion_report).transpose())
print("\nKernel SVM (RBF) (Fashion):")
print(pd.DataFrame(svm_rbf_fashion_report).transpose())
print("\nNeural Network (MLP) (Fashion):")
print(pd.DataFrame(mlp_fashion_report).transpose())